# 02 · Indicators for Volatile Markets: BTC/USDT and ETH/USDT

Notebook 01 found that crypto **volatility is predictable but direction mostly is not**, with fat tails and sudden jumps.
This notebook takes that further with a set of more advanced indicators, each chosen for a reason, and asks one honest question about each:

> **What does this indicator actually tell us about the next week: the *direction* of the move, its *size*, or nothing?**

| Family | Indicators | Why they suit volatile markets |
|---|---|---|
| 1. Volatility estimators | close-to-close, Parkinson, Garman–Klass, Rogers–Satchell, Yang–Zhang, ATR | Use the whole candle (high/low/open), not just the close: more information per bar |
| 2. Realised measures (hourly → daily) | realised variance, up/down semivariance, bipower variation, jump share, realised skew/kurtosis | Split a day's risk into "good" vs "bad" and smooth vs jump moves |
| 3. Volatility regime | volatility percentile, short/long term structure, volatility of volatility, Bollinger–Keltner squeeze | Say whether today is calm or stormy *relative to history* |
| 4. Trend vs chop | ADX/DMI, Choppiness Index, efficiency ratio, Hurst exponent, Supertrend | Say whether trend-following should work right now |
| 5. Stretch | RSI, stochastic RSI, Bollinger %B, z-score, drawdown from the 90-day high | Measure how far price has run |
| 6. Flow and liquidity | Chaikin Money Flow, relative volume, taker-buy imbalance, Amihud illiquidity | Who is pushing the price, and how thin the market is |

**Rules followed**
- Data: **2021-01-01 → 2024-12-31** (2024 counts as training data since protocol v4). **2025 is not loaded anywhere.**
- Every indicator is written in-house in `src/features/indicators.py` and is **causal**: its value on day *t* uses data up to day *t*'s close (checked by truncation tests in `tests/test_indicators.py`).
- Future returns and volatility are built with negative shifts **only as research targets**, never as inputs.
- Statistics here are **descriptive**. Any number that later becomes a strategy parameter must be re-derived causally.

## 0. Setup

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

ROOT = Path.cwd().parent            # the notebook lives in notebooks/
sys.path.insert(0, str(ROOT))
from src.data.loader import load_market
from src.features import indicators as ind

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 4)
pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.width", 160)

FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
ASSETS = ["BTCUSDT", "ETHUSDT"]
COLORS = {"BTCUSDT": "#F7931A", "ETHUSDT": "#627EEA"}
YEARS = [2021, 2022, 2023, 2024]
HORIZON = 7                          # we judge every indicator on the next 7 days

## 1. Data and research targets

We load daily bars (for the indicators) and hourly bars (for the realised measures) from the train (2021–23) and validation (2024) splits.

Two **targets**, both for the 7 days *after* day *t* (days t+1 … t+7):
- `fwd_ret`: the 7-day log return, which measures **direction**;
- `fwd_vol`: annualised realised volatility from hourly returns, which measures **size**.

Because a signal from day *t* can only be traded from day *t+1*'s open, the targets start at *t+1*.

In [ ]:
def load_years(symbol, timeframe):
    return pd.concat([load_market(symbol, timeframe, split) for split in ["train", "val"]])


daily = {s: load_years(s, "1d") for s in ASSETS}
hourly = {s: load_years(s, "1h") for s in ASSETS}
for s in ASSETS:
    print(s, daily[s].index.min().date(), "->", daily[s].index.max().date(), f"{len(daily[s])} days, {len(hourly[s])} hours")

In [ ]:
realised = {s: ind.realised_measures(hourly[s]).reindex(daily[s].index) for s in ASSETS}


def forward_targets(d, rm, h=HORIZON):
    # Research targets only: they look into the future on purpose and are never strategy inputs.
    log_ret = np.log(d["close"]).diff()
    fwd_ret = log_ret[::-1].rolling(h).sum()[::-1].shift(-1)
    fwd_vol = np.sqrt(rm["rv"][::-1].rolling(h).mean()[::-1].shift(-1) * 365)
    return pd.DataFrame({"fwd_ret": fwd_ret, "fwd_vol": fwd_vol})


targets = {s: forward_targets(daily[s], realised[s]) for s in ASSETS}
targets["BTCUSDT"].describe().T

**How we score an indicator.** For each year separately we compute the **Spearman rank correlation** (the "information coefficient", IC) between the indicator on day *t* and each target.
- Ranks are used because crypto returns are fat-tailed: a single crash should not decide the result.
- Doing it **per year** is the robustness test. An effect we can trust must keep the **same sign in at least 3 of 4 years** (2021 bull, 2022 bear, 2023 recovery, 2024 range).
- Rough scale: with ~365 overlapping 7-day windows, an |IC| below ~0.10 in a single year is indistinguishable from noise.

In [ ]:
def yearly_ic(x, target):
    # Spearman correlation of an indicator with a target, separately for each calendar year.
    df = pd.concat([x.rename("x"), target.rename("y")], axis=1).dropna()
    return pd.Series({y: g["x"].corr(g["y"], method="spearman") for y, g in df.groupby(df.index.year)})


def ic_table(features, target):
    table = pd.DataFrame({name: yearly_ic(col, target) for name, col in features.items()}).T
    table["same_sign_years"] = (np.sign(table[YEARS]) == np.sign(table[YEARS].mean(axis=1)).values[:, None]).sum(axis=1)
    return table

## 2. Volatility estimators: how much does the whole candle help?

The textbook volatility uses only closes. But every daily candle also contains the **high, low and open**, which tell us how far price travelled during the day.
- **Parkinson** uses the high–low range. For a random walk it needs ~5× fewer days than close-to-close for the same precision.
- **Garman–Klass** adds open and close.
- **Rogers–Satchell** stays unbiased when the market trends (drift), and crypto trends hard.
- **Yang–Zhang** combines the gap between sessions with Rogers–Satchell.
- **ATR**, the trader's version, is in price units, so we divide it by the close.
- **Realised vol (hourly)** sums squared hourly returns: the most data-rich of all.

All use a 30-day window (7 days for realised) and are annualised with √365.

In [ ]:
def vol_estimators(d, rm):
    return pd.DataFrame({
        "close_to_close": ind.close_to_close_vol(d["close"], 30),
        "parkinson": ind.parkinson_vol(d, 30),
        "garman_klass": ind.garman_klass_vol(d, 30),
        "rogers_satchell": ind.rogers_satchell_vol(d, 30),
        "yang_zhang": ind.yang_zhang_vol(d, 30),
        "atr_pct": ind.atr(d, 14) / d["close"] * np.sqrt(365),
        "realised_7d": np.sqrt(rm["rv"].rolling(7).mean() * 365),
    })


vols = {s: vol_estimators(daily[s], realised[s]) for s in ASSETS}
vols["BTCUSDT"].describe().T[["mean", "50%", "min", "max"]]

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for ax, s in zip(axes, ASSETS):
    vols[s][["close_to_close", "parkinson", "yang_zhang", "realised_7d"]].plot(ax=ax, lw=1)
    ax.set_title(f"{s}: annualised volatility estimators")
    ax.set_ylabel("annualised vol")
plt.tight_layout()
plt.savefig(FIG / "ind_vol_estimators.png", dpi=120)
plt.show()

In [ ]:
# Which estimator best predicts next week's realised volatility?
for s in ASSETS:
    print(s)
    display(ic_table(vols[s], targets[s]["fwd_vol"]).round(2))

**What it shows**
- Every estimator predicts next week's volatility, with a positive IC in **every** year (mostly +0.2 to +0.6). Volatility clusters: a stormy month is followed by a stormy week.
- **Realised volatility from hourly data is the best or tied-best forecaster in 6 of 8 coin-years** (e.g. BTC 2021: 0.62 vs 0.43 for close-to-close; ETH 2021: 0.70 vs 0.50). Twenty-four observations a day beat one.
- **ATR% (14-day, exponentially weighted) is a close second.** It is not cleverer; it simply reacts faster than the 30-day windows. *How recent* the data is matters as much as which formula is used.
- At the same 30-day window, the range-based estimators (Parkinson, Yang–Zhang) do **not** clearly beat close-to-close. Parkinson wins 2 of 4 years for BTC and 3 of 4 for ETH, so the textbook "5× more efficient" advantage shrinks on real crypto data with jumps and trends.
- With daily candles only, the practical choice is a **short-window** range-based estimator or ATR.
- In the chart, all estimators agree on *when* markets were stormy (May 2021, June and November 2022, March 2023, August 2024). They differ in how noisy they are, and close-to-close is the jumpiest.
- Range estimators read slightly higher than close-to-close (BTC mean ~0.62 vs ~0.58). Large intraday swings that end near the open are invisible to closes but are real risk for a stop-loss. (ATR% is on a different scale because the true range is not a standard deviation.)

## 3. Inside the day: semivariance and jumps (from hourly data)

Total realised variance hides *how* the day moved. We split it two ways:
1. **Up vs down semivariance**: variance from rising hours vs falling hours. Finance research (Patton & Sheppard, 2015) finds "bad" volatility predicts future risk better than "good" volatility.
2. **Smooth vs jump**:
   - **Bipower variation** multiplies consecutive absolute returns, so a single huge hour barely moves it. It measures the *smooth* (continuous) part of variance.
   - `jump_share = (RV − BV) / RV` is the fraction of the day's variance that came from sudden jumps.

In [ ]:
realised["BTCUSDT"][["rv", "rs_up", "rs_down", "bv", "jump_share", "signed_jump", "rskew", "rkurt"]].describe().T[["mean", "50%", "max"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for s in ASSETS:
    realised[s]["jump_share"].plot.hist(bins=60, alpha=0.5, ax=axes[0], color=COLORS[s], label=s, density=True)
    realised[s]["rkurt"].clip(upper=20).plot.hist(bins=60, alpha=0.5, ax=axes[1], color=COLORS[s], label=s, density=True)
axes[0].set_title("Share of daily variance from jumps")
axes[1].set_title("Realised kurtosis of hourly returns (3 = normal)")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.savefig(FIG / "ind_jumps.png", dpi=120)
plt.show()

Which part of today's risk **carries forward**? We regress next week's log realised variance on the last 7 days' log components (a HAR-style model; HAC standard errors because the 7-day windows overlap).

In [ ]:
def log_mean7(x):
    return np.log(x.rolling(7).mean() + 1e-10)


def har_fit(rm, target, parts):
    X = pd.DataFrame({name: log_mean7(series) for name, series in parts(rm).items()})
    y = np.log(target ** 2 / 365).rename("y")
    data = pd.concat([X, y], axis=1).dropna()
    fit = sm.OLS(data["y"], sm.add_constant(data[X.columns])).fit(cov_type="HAC", cov_kwds={"maxlags": 7})
    return pd.DataFrame({"coef": fit.params, "t_stat": fit.tvalues}).drop("const")


smooth_vs_jump = lambda rm: {"smooth (bipower)": rm["bv"], "jump (RV-BV)": (rm["rv"] - rm["bv"]).clip(lower=0)}
down_vs_up = lambda rm: {"downside semivar": rm["rs_down"], "upside semivar": rm["rs_up"]}

for s in ASSETS:
    print(s)
    display(pd.concat({"smooth vs jump": har_fit(realised[s], targets[s]["fwd_vol"], smooth_vs_jump),
                       "down vs up": har_fit(realised[s], targets[s]["fwd_vol"], down_vs_up)}).round(3))

**What it shows**
- About **14% of daily variance comes from jumps** on an average day for both coins. Some days are dominated by one jump: the long right tail of the histogram, up to 85% (BTC).
- Hourly returns have realised kurtosis averaging ~4.7 (median ~3.9; normal = 3). Even *within* a single day the moves are fat-tailed.
- **Smooth volatility persists; jumps do not.** The coefficient on the smooth part (~0.58) is several times the jump part's (~0.08 BTC, ~0.12 ETH).
  - A sudden one-hour crash is mostly a one-off.
  - A day of steady, heavy movement predicts a heavy week.
  - For risk sizing, react to the smooth part and partly "forgive" isolated jumps.
- **Bad volatility predicts more than good volatility.** Downside semivariance gets about 2× the weight of upside semivariance for BTC (0.45 vs 0.21), and 0.41 vs 0.29 for ETH.
  - Volatility born in a sell-off keeps going; volatility from a rally fades faster.
  - This is the "inverse leverage effect" from notebook 01 seen from another angle.

## 4. Volatility regime: calm or stormy *compared to normal*?

Absolute levels like "60% vol" mean different things in 2021 and 2024. Regime indicators compare today with the coin's own history:
- **Volatility percentile**: rank of 14-day Parkinson vol within the trailing 365 days (expanding until a year is available). It runs from 0 (calmest of the year) to 1 (stormiest).
- **Term structure**: `log(vol 7d / vol 60d)`. Above 0 means short-term vol is above long-term vol, i.e. a spike in progress.
- **Volatility of volatility**: standard deviation of daily changes in log-vol over 30 days, i.e. how unstable the risk itself is.
- **Bollinger bandwidth percentile** and the **squeeze** (Bollinger Bands inside Keltner Channels). The squeeze is a classic trader's signal that the market is "coiling" before a big move.

In [ ]:
def regime_features(d):
    bb = ind.bollinger(d["close"], 20)
    return pd.DataFrame({
        "vol_percentile": ind.rolling_percentile(ind.parkinson_vol(d, 14)),
        "term_structure": np.log(ind.parkinson_vol(d, 7) / ind.parkinson_vol(d, 60)),
        "vol_of_vol": ind.vol_of_vol(ind.parkinson_vol(d, 7), 30),
        "bb_width_pct": ind.rolling_percentile(bb["bb_width"]),
        "squeeze": ind.squeeze(d),
    })


regime = {s: regime_features(daily[s]) for s in ASSETS}
for s in ASSETS:
    print(s, f"squeeze on {regime[s]['squeeze'].mean():.0%} of days")
    display(ic_table(regime[s], targets[s]["fwd_vol"]).round(2))

The raw IC mixes two things: *"vol is high now"* and *"vol will change"*. To see what an indicator adds **beyond current volatility**, we remove the part of next week's vol explained by the last 7 days' realised vol and correlate the indicator with what is left.

This fit uses the whole 2021–24 sample. It is a descriptive check, not a forecast.

In [ ]:
def beyond_current_vol(features, s):
    current = np.log(vols[s]["realised_7d"])
    future = np.log(targets[s]["fwd_vol"])
    data = pd.concat([current.rename("c"), future.rename("f")], axis=1).dropna()
    residual = sm.OLS(data["f"], sm.add_constant(data["c"])).fit().resid
    return ic_table(features, residual)


for s in ASSETS:
    print(s, "IC with the part of next week's vol NOT explained by current vol")
    display(beyond_current_vol(regime[s], s).round(2))

In [ ]:
def squeeze_event_study(s):
    # Average log change from current to next-week vol, inside / outside a squeeze and on the day it releases.
    sq = regime[s]["squeeze"]
    change = np.log(targets[s]["fwd_vol"] / vols[s]["realised_7d"])
    released = (sq.shift(1) == 1) & (sq == 0)
    abs_move = targets[s]["fwd_ret"].abs()
    return pd.DataFrame({
        "days": [int((sq == 0).sum()), int((sq == 1).sum()), int(released.sum())],
        "vol_change_next_week": [change[sq == 0].mean(), change[sq == 1].mean(), change[released].mean()],
        "abs_7d_move": [abs_move[sq == 0].mean(), abs_move[sq == 1].mean(), abs_move[released].mean()],
        "mean_7d_return": [targets[s]["fwd_ret"][m].mean() for m in [sq == 0, sq == 1, released]],
    }, index=["no squeeze", "in squeeze", "squeeze release day"])


for s in ASSETS:
    print(s)
    display(squeeze_event_study(s))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for ax, s in zip(axes, ASSETS):
    close = daily[s]["close"]
    ax.plot(close.index, close, color=COLORS[s], lw=1)
    sq = regime[s]["squeeze"] == 1
    ax.scatter(close.index[sq], close[sq], s=6, color="black", label="squeeze on", zorder=3)
    ax.set_yscale("log")
    ax.set_title(f"{s}: days in a Bollinger-Keltner squeeze")
    ax.legend()
plt.tight_layout()
plt.savefig(FIG / "ind_squeeze.png", dpi=120)
plt.show()

**What it shows**
- **Volatility mean-reverts around a clustering core.** Raw ICs of vol percentile and term structure are strongly positive, because high vol now means high vol next week.
  - Once current volatility is accounted for, the signs flip negative (vol percentile in 8 of 8 coin-years, Bollinger-width percentile in 8 of 8).
  - An unusually stormy market calms down faster than its current level suggests, and an unusually calm one heats up.
  - Vol-of-vol behaves the same way: unstable volatility tends to settle.
- **The squeeze works, but only for size, not direction.**
  - Inside a squeeze (15–17% of days), next week's vol is on average ~10–14% *higher* than current vol (log change +0.09 BTC, +0.13 ETH). Outside a squeeze it is ~3% lower.
  - Beyond-current-vol IC is positive in 7 of 8 coin-years. The "coiled spring" story holds in the sense that volatility expands.
  - Mean 7-day returns after a squeeze or release are small and inconsistent. The squeeze does **not** tell us which way the breakout goes.
  - For BTC, the absolute 7-day move after a release is larger (8.1% vs 6.1% outside a squeeze). For ETH it is not (6.7% vs 8.0%).
- **Practical reading:**
  - A squeeze is a warning to **shrink position size or widen stops ahead of a volatility expansion**.
  - It is also a cue for a breakout strategy to *get ready*. The direction must come from somewhere else (trend).

## 5. Trend or chop? Does trend-following work right now?

Our earlier strategies lost most of their money in sideways, choppy markets. These indicators try to detect such markets in advance:
- **ADX** (Wilder): trend *strength* regardless of direction; above 25 is the usual "trending" threshold. **+DI − −DI** gives the direction.
- **Choppiness Index**: path length (sum of true ranges) relative to the net range. Above 61.8 means chop and below 38.2 means trend (traditional Fibonacci levels).
- **Efficiency ratio**: |net move| ÷ sum of |daily moves| over 30 days. It is 1 for a straight line and ~0 for back-and-forth.
- **Hurst exponent** (rescaled range, 128 days): above 0.5 suggests persistent trends, below 0.5 suggests mean reversion.
- **Supertrend** (10, 3): a popular ATR-based trailing line, giving +1 for an uptrend and −1 for a downtrend.

To test them directly, we define a simple **trend-following payoff**: `sign(+DI − −DI) × next-week return`. It is positive when betting on the current direction would have paid.

In [ ]:
def trend_features(d):
    dmi = ind.adx(d, 14)
    return pd.DataFrame({
        "adx": dmi["adx"],
        "di_spread": dmi["plus_di"] - dmi["minus_di"],
        "choppiness": ind.choppiness(d, 14),
        "efficiency_ratio": ind.efficiency_ratio(d["close"], 30),
        "hurst": ind.rolling_hurst(d["close"], 128),
        "supertrend": ind.supertrend(d, 10, 3.0),
    })


trend = {s: trend_features(daily[s]) for s in ASSETS}
for s in ASSETS:
    print(s, "- IC with next-week RETURN")
    display(ic_table(trend[s], targets[s]["fwd_ret"]).round(2))

In [ ]:
def payoff_by_bucket(s, column, bins, labels):
    # Average trend-following payoff in each bucket of an indicator, per year.
    payoff = np.sign(trend[s]["di_spread"]) * targets[s]["fwd_ret"]
    bucket = pd.cut(trend[s][column], bins, labels=labels)
    table = payoff.groupby([payoff.index.year, bucket]).mean().unstack()
    table.loc["share of days"] = bucket.value_counts(normalize=True)[labels].values
    return table


for s in ASSETS:
    print(s, "- trend-following payoff by Choppiness regime")
    display(payoff_by_bucket(s, "choppiness", [0, 38.2, 61.8, 100], ["trending (<38.2)", "middle", "choppy (>61.8)"]).round(3))
    print(s, "- trend-following payoff by ADX")
    display(payoff_by_bucket(s, "adx", [0, 20, 30, 100], ["ADX<20", "20-30", "ADX>30"]).round(3))

**Is the Hurst exponent telling us anything?** The rescaled-range estimator is known to be biased upwards in short windows. We test it by computing Hurst on the **same returns shuffled in random order**, which has no trend by construction. If the real series scores about the same, the indicator is showing its own bias, not market structure.

In [ ]:
rng = np.random.default_rng(0)
rows = {}
for s in ASSETS:
    returns = np.log(daily[s]["close"]).diff().dropna().to_numpy()
    shuffled_price = pd.Series(np.exp(np.cumsum(rng.permutation(returns))))
    rows[s] = {"hurst on real data": trend[s]["hurst"].mean(),
               "hurst on shuffled returns (no memory)": ind.rolling_hurst(shuffled_price, 128).mean()}
pd.DataFrame(rows).T

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
s = "BTCUSDT"
axes[0].plot(daily[s]["close"], color=COLORS[s], lw=1)
axes[0].set_yscale("log")
axes[0].set_title("BTC price")
axes[1].plot(trend[s]["adx"], lw=1)
axes[1].axhline(25, color="grey", ls="--")
axes[1].set_title("ADX(14): trend strength")
axes[2].plot(trend[s]["choppiness"], lw=1, color="purple")
for level in (38.2, 61.8):
    axes[2].axhline(level, color="grey", ls="--")
axes[2].set_title("Choppiness Index(14): above 61.8 chop, below 38.2 trend")
plt.tight_layout()
plt.savefig(FIG / "ind_trend_chop.png", dpi=120)
plt.show()

**What it shows**
- **No trend indicator reliably predicts next week's direction.** ADX, DI spread and efficiency ratio have ICs that flip sign between years.
  - Supertrend is a good example. Its IC was positive only in the 2021 bull and **negative in 2022–24 for both coins**: being in "uptrend" was followed by *lower* returns.
  - On daily bars it flips late, after the move. In range-bound years the dips it calls "downtrend" were buying opportunities.
  - Choppiness has a small negative tilt in most years (choppy now, slightly weaker next week), but the ICs are mostly below 0.1.
- **ADX is not a reliable "trend-following works" switch.**
  - Trend-following payoff is not higher when ADX > 30.
  - BTC's best ADX bucket in 2021 was ADX < 20, and ETH's worst in 2021 was ADX > 30.
  - High ADX mostly flags a **large move that has already happened** (its link with next-week vol is positive in 2022–24).
- **The Choppiness Index is the most useful filter here.**
  - When Choppiness < 38.2 (clean trend), trend-following paid in **4 of 4 years for BTC** (+4.8%, +0.9%, +1.1%, +2.5% average 7-day payoff) and 3 of 4 for ETH.
  - The catch: true "trending" readings occur on only ~12–14% of days. It identifies the *good* trending windows rather than switching strategies on and off.
  - Its "choppy" reading (>61.8) does **not** reliably predict losses, so it should be used to add conviction, not to veto.
- **The Hurst exponent is a fancy indicator that shows nothing on daily crypto.**
  - The real data averages ~0.59, which looks like "trending".
  - Shuffled returns, which have no memory at all, score almost the same: ~0.59 BTC, ~0.57 ETH.
  - The "persistence" is estimator bias. We drop Hurst rather than present a misleading number.

## 6. Stretch: is the market overbought or oversold?

Classic oscillators assume prices that ran too far snap back:
- **RSI(14)**: above 70 is overbought and below 30 is oversold.
- **Stochastic RSI**: where RSI sits in its own 14-day range.
- **Bollinger %B**: where the close sits between the bands (above 1 means outside the top band).
- **Z-score(50)**: distance from the 50-day mean in standard deviations.
- **Drawdown from the 90-day high**: how deep the current sell-off is.

In [ ]:
def stretch_features(d):
    return pd.DataFrame({
        "rsi": ind.rsi(d["close"], 14),
        "stoch_rsi": ind.stoch_rsi(d["close"], 14),
        "bollinger_pct_b": ind.bollinger(d["close"], 20)["bb_pct_b"],
        "zscore_50": ind.zscore(d["close"], 50),
        "drawdown_90d": ind.drawdown_from_high(d["close"], 90),
    })


stretch = {s: stretch_features(daily[s]) for s in ASSETS}
for s in ASSETS:
    print(s, "- IC with next-week RETURN")
    display(ic_table(stretch[s], targets[s]["fwd_ret"]).round(2))

In [ ]:
def rsi_extremes(s):
    r, f = stretch[s]["rsi"], targets[s]["fwd_ret"]
    rows = {}
    for label, mask in {"RSI < 30": r < 30, "RSI > 70": r > 70, "all days": r.notna()}.items():
        g = f[mask].groupby(f[mask].index.year)
        rows[label] = pd.concat({"mean 7d return": g.mean(), "days": g.size()}, axis=1).unstack()
    return pd.DataFrame(rows).T


for s in ASSETS:
    print(s)
    display(rsi_extremes(s).round(3))

In [ ]:
def drawdown_buckets(s):
    bucket = pd.cut(stretch[s]["drawdown_90d"], [-1, -0.4, -0.2, -0.05, 0.01],
                    labels=["deeper than -40%", "-40% to -20%", "-20% to -5%", "within 5% of the high"])
    return targets[s].groupby(bucket).mean().join(bucket.value_counts(normalize=True).rename("share of days"))


for s in ASSETS:
    print(s)
    display(drawdown_buckets(s).round(3))

**What it shows**
- **Oscillators do not give a stable direction signal on daily crypto.** RSI, stochastic RSI, %B and z-score all have yearly ICs mostly within ±0.1, with signs that change from year to year.
- **Drawdown depth has a weak, fairly consistent dip-rebound tilt.**
  - Its IC is negative in 4 of 4 BTC years and 3 of 4 ETH years: the deeper below the 90-day high, the better next week tends to be.
  - The effect is small (|IC| mostly 0.05–0.15, the largest in 2022) and the bucket table below is not monotonic.
- **"Overbought" depends on the regime.**
  - In bull years (2023, 2024), RSI > 70 was followed by *further gains* (BTC +2.2% and +4.4% over 7 days, from 68 and 53 days).
  - In the 2022 bear, the few RSI > 70 days were followed by losses. Selling "overbought" crypto in a bull market costs money.
- **"Oversold" has a weak bounce tendency** (RSI < 30 followed by positive 7-day returns for BTC in 2022, 2023 and 2024). But there were only 4–28 such days a year, which is too few to trust on its own.
- **Drawdown depth is a risk gauge, not a timing tool.**
  - More than 40% below the 90-day high, next-week vol is very high (BTC ~0.79 vs ~0.54 near the high; ETH ~1.0).
  - Average returns there are not reliably negative. Deep drawdowns mean *bigger swings both ways*, so size must shrink even if the bias is to buy.

## 7. Flow and liquidity: who is pushing the price?

- **Chaikin Money Flow (20)**: volume-weighted position of each close inside its daily range. Above 0 suggests accumulation and below 0 distribution.
- **Relative volume**: log of today's USDT volume relative to its 30-day median.
- **Taker-buy imbalance z-score**: share of volume from aggressive buyers versus its own 30-day norm (this needs Binance's taker fields).
- **Amihud illiquidity**: |return| per $1M traded. When this is high, the market is thin and moves easily.

In [ ]:
def flow_features(d):
    return pd.DataFrame({
        "chaikin_money_flow": ind.chaikin_money_flow(d, 20),
        "relative_volume": ind.relative_volume(d, 30),
        "taker_imbalance_z": ind.taker_imbalance_z(d, 30),
        "amihud_illiquidity": np.log(ind.amihud(d, 30)),
    })


flow = {s: flow_features(daily[s]) for s in ASSETS}
for s in ASSETS:
    print(s, "- IC with next-week RETURN")
    display(ic_table(flow[s], targets[s]["fwd_ret"]).round(2))
    print(s, "- IC with next-week VOL, beyond current vol")
    display(beyond_current_vol(flow[s], s).round(2))

**What it shows**
- **Chaikin Money Flow and taker imbalance carry no reliable direction signal at a weekly horizon.** Their ICs are near zero with changing signs. Aggressive buying today does not predict next week's price; at most it is absorbed within hours.
- **Volume surges warn of volatility.** Relative volume predicts next week's vol *beyond* current vol in 7 of 8 coin-years (BTC +0.17, +0.24, +0.05, +0.05; ETH +0.15, +0.13, −0.01, +0.05). Unusual activity tends to come before larger moves, so it is a useful early-warning input for risk sizing.
  - Two smaller patterns: strong taker *buying* is followed by slightly *lower* vol (negative in 3 of 4 years for both coins), and for ETH positive money flow is followed by higher vol (4 of 4).
- **Amihud illiquidity is the most consistent direction signal in this notebook,** and also the one to be most careful with.
  - Its return IC is ≥ 0 in all 8 coin-years (BTC 0.30, 0.00, 0.07, 0.06; ETH 0.35, 0.20, 0.22, 0.04). Thin markets (big moves per dollar traded) were followed by better weeks, consistent with the *liquidity premium* in finance research.
  - But the level drifts with exchange-wide volume, including the 2022–23 Binance zero-fee BTC volume distortion (notebook 01). For BTC it is almost entirely 2021.
  - Treat it as a **candidate to re-test causally**, not a finding.

## 8. Scorecard: every indicator on one page

All indicators side by side. The left panel shows the yearly IC with next-week **return** (direction). The right panel shows the yearly IC with next-week vol **beyond current vol** (genuinely new information about size).

Blue and red cells that stay the same colour across all four years are the signals worth building on.

In [ ]:
def all_features(s):
    rm = realised[s]
    micro = pd.DataFrame({
        "semivar_ratio (down/up)": np.log(rm["rs_down"].rolling(7).sum() / rm["rs_up"].rolling(7).sum()),
        "jump_share": rm["jump_share"].rolling(7).mean(),
        "realised_skew": rm["rskew"].rolling(7).mean(),
        "realised_kurtosis": rm["rkurt"].rolling(7).mean(),
    })
    return pd.concat([regime[s], trend[s], stretch[s], flow[s], micro], axis=1)


fig, axes = plt.subplots(2, 2, figsize=(14, 16))
for row, s in enumerate(ASSETS):
    feats = all_features(s)
    direction = ic_table(feats, targets[s]["fwd_ret"])[YEARS]
    size = beyond_current_vol(feats, s)[YEARS]
    for ax, table, title in [(axes[row, 0], direction, "next-week RETURN"), (axes[row, 1], size, "next-week VOL beyond current vol")]:
        sns.heatmap(table, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-0.4, vmax=0.4, ax=ax, cbar=False)
        ax.set_title(f"{s}: IC with {title}")
plt.tight_layout()
plt.savefig(FIG / "ind_scorecard.png", dpi=110)
plt.show()

**Reading the scorecard**
- The **left side is mostly pale and patchy.** Of 25+ indicators, only Amihud illiquidity (and, weakly, drawdown depth and choppiness) keeps a mostly consistent sign for both coins, and even those are small or caveated. This repeats notebook 01's central finding with much fancier tools: in crypto, direction is hard and indicator-based direction calls are mostly noise.
- The **right side has consistent colour.** Measured beyond current volatility:
  - Calm-relative-to-history readings (low vol percentile, low Bollinger-width percentile, squeeze on) predict a volatility **rise**.
  - Volume surges and downside-dominated variance (BTC) predict a **rise**.
  - Unstable vol-of-vol and spikes in the term structure predict a **fall**.
- **ETH has its own asymmetry.** Bullish readings (DI spread, Supertrend up, RSI, z-score, money flow) predict *higher* ETH volatility beyond current vol in 4 of 4 years (e.g. DI spread +0.19, +0.20, +0.09, +0.23).
  - ETH rallies are violent: the "inverse leverage effect" from notebook 01.
  - For BTC the extra risk comes from the other side: downside-dominated variance (semivariance ratio positive in 4 of 4 years).
- Spiky intraday weeks (high realised kurtosis) were followed by slightly weaker ETH returns in 4 of 4 years, but the ICs are small (−0.05 to −0.13).

**Indicators are risk instruments first and timing instruments second.**

## 9. A regime map that combines the useful pieces

Using only causal inputs, we split days into four regimes:
- **Volatility:** high if the volatility percentile is above 0.5, i.e. stormier than the median day of the past year.
- **Trendiness:** trending if the 30-day efficiency ratio is above its own *expanding* median, which uses only past days.

For each regime we look at next-week volatility and the payoff of a plain 30-day momentum bet, `sign(30-day return) × next-week return`.

In [ ]:
def regime_map(s):
    high_vol = regime[s]["vol_percentile"] > 0.5
    er = trend[s]["efficiency_ratio"]
    trending = er > er.expanding(min_periods=90).median()
    label = np.select([high_vol & trending, high_vol & ~trending, ~high_vol & trending],
                      ["high-vol trend", "high-vol chop", "low-vol trend"], "low-vol chop")
    valid = regime[s]["vol_percentile"].notna() & er.expanding(min_periods=90).median().notna()
    return pd.Series(label, index=daily[s].index).where(valid)


def momentum_payoff(s):
    return np.sign(np.log(daily[s]["close"]).diff(30)) * targets[s]["fwd_ret"]


for s in ASSETS:
    reg = regime_map(s)
    pay = momentum_payoff(s)
    summary = pd.DataFrame({"share of days": reg.value_counts(normalize=True),
                            "next-week vol": targets[s]["fwd_vol"].groupby(reg).mean(),
                            "momentum payoff (7d)": pay.groupby(reg).mean()})
    print(s)
    display(summary.round(3))
    display(pay.groupby([pay.index.year, reg]).mean().unstack().round(3))

In [ ]:
palette = {"high-vol trend": "#d62728", "high-vol chop": "#ff9896", "low-vol trend": "#2ca02c", "low-vol chop": "#98df8a"}
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for ax, s in zip(axes, ASSETS):
    close, reg = daily[s]["close"], regime_map(s)
    for name, colour in palette.items():
        mask = reg == name
        ax.scatter(close.index[mask], close[mask], s=5, color=colour, label=name)
    ax.set_yscale("log")
    ax.set_title(f"{s}: price coloured by volatility x trendiness regime")
    ax.legend(markerscale=3, loc="upper left")
plt.tight_layout()
plt.savefig(FIG / "ind_regime_map.png", dpi=120)
plt.show()

**What it shows**
- **Momentum works best when the market is calm and trending.**
  - In the *low-vol trend* regime, the 30-day momentum payoff was positive in **4 of 4 years for both coins** (BTC average +1.4% per 7 days, ETH +2.3%).
  - In the two *high-vol* regimes it averaged roughly zero or negative.
  - ETH's high-vol trend regime in 2021 was the worst cell (−15%): momentum bought into the May 2021 crash.
- This matches what an options trader would expect. A calm, orderly trend is a market where buyers keep coming. A violent trend is often a blow-off or capitulation that reverses.
- High-volatility regimes cover ~38–42% of days. Being **smaller** in them costs little in missed trend profit and saves a lot in drawdown.
- In the chart, the May 2021 and May–June 2022 crashes sit in red (high-vol), and the steady 2023 climbs in green (low-vol).
- The map does **not** call tops: both coins' November 2021 peaks were reached in calm, green conditions. It tells us *how* to size a trend, not when it ends.

## 10. Indicators from a cited reference: the Inter IIT 13.0 Team 67 report

`docs/references/Zelta_Final_Report (1).pdf` is a previous Inter IIT team's BTC/ETH submission. Its ETH strategy combines several indicators, and its BTC agent uses others in its state. We re-implement each one in-house and test it **the same way as every indicator above** (yearly IC with next-week return and with next-week volatility beyond current volatility). Then we check the specific rules the report built on them.

| Indicator | How the report used it | Our version |
|---|---|---|
| ATR % of price | trade only when hourly ATR < 1% of price; exit when > 2.5% | same, on hourly bars, sampled at each day's close |
| BTC–ETH rolling correlation | trade ETH only when correlation > 0.6 | correlation of hourly returns over the last 7 days (168 h) |
| EMA ribbon 7/14/28 | +1 bullish order, −1 bearish order (RL state) | same |
| Aroon (25) | trend direction (RL state) | Aroon oscillator (up − down) |
| Heikin-Ashi candles | smoothed reference price | same (causal: each candle uses only the current and previous bar) |
| Gaussian filter | smoothed reference price | **one-sided** Gaussian (the usual centred filter uses future bars and is forbidden) |
| Kalman filter | best reference price for CUSUM | causal filter only (no smoother): its slope and the price's deviation from it |
| CUSUM regimes | BTC bull/bear regime guiding ETH | two-sided CUSUM on the Kalman reference |
| Markov regime switching | experimented with | 2-regime switching-variance model, refit yearly on past data, **filtered** probability |
| PCA + K-means volatility clusters | experimented with | 3 clusters on volatility features, refit yearly on past data |
| RSI 75 / 35 | RL state thresholds | same thresholds |
| Hurst exponent | trade only when H > 0.5 | already tested in section 5: estimator bias, no information |

In [ ]:
corr_btc_eth = ind.rolling_corr_hourly(hourly["BTCUSDT"], hourly["ETHUSDT"]).reindex(daily["BTCUSDT"].index)


def reference_features(s):
    d, h = daily[s], hourly[s]
    hourly_atr = ind.atr_pct(h, 14)
    by_day = hourly_atr.index.floor("1D")
    kal = ind.kalman_trend(d["close"])
    vol_inputs = pd.DataFrame({"parkinson_14": ind.parkinson_vol(d, 14), "atr_pct": ind.atr_pct(d),
                               "rv_7d": realised[s]["rv"].rolling(7).mean()})
    return pd.DataFrame({
        "hourly_atr_pct": hourly_atr.groupby(by_day).last().reindex(d.index),
        "share_hours_atr_below_1pct": (hourly_atr < 0.01).groupby(by_day).mean().reindex(d.index),
        "btc_eth_corr_7d": corr_btc_eth,
        "ema_ribbon": ind.ema_ribbon(d["close"]),
        "aroon_osc": ind.aroon(d, 25)["aroon_osc"],
        "heikin_ashi_streak": ind.heikin_ashi(d)["ha_streak"],
        "gaussian_slope": np.log(ind.causal_gaussian(d["close"])).diff(),
        "kalman_slope": kal["kalman_slope"],
        "kalman_deviation": kal["kalman_dev"],
        "cusum_bullish": ind.cusum_state(d["close"]),
        "markov_p_high_vol": ind.markov_switching_prob(np.log(d["close"]).diff()),
        "vol_cluster": ind.vol_clusters(vol_inputs),
    })


reference = {s: reference_features(s) for s in ASSETS}
reference["BTCUSDT"].describe().T[["mean", "50%", "min", "max"]]

In [ ]:
for s in ASSETS:
    print(s, "- IC with next-week RETURN")
    display(ic_table(reference[s], targets[s]["fwd_ret"]).round(2))
    print(s, "- IC with next-week VOL, beyond current vol")
    display(beyond_current_vol(reference[s], s).round(2))

### 10.1 The report's own rules, checked one by one
Each rule tested on 2021–2024 with the same 30-day momentum payoff used in section 9 (`sign(30-day return) × next-week return`). That is the payoff a trend trade would have earned.

In [ ]:
rows = {}
for s in ASSETS:
    f, t = reference[s], targets[s]
    payoff = np.sign(np.log(daily[s]["close"]).diff(30)) * t["fwd_ret"]
    rules = {   # (indicator, condition); days where the indicator does not exist yet are left out
        "hourly ATR < 1% on most hours": ("share_hours_atr_below_1pct", lambda x: x > 0.5),
        "hourly ATR > 2.5% (report: exit)": ("hourly_atr_pct", lambda x: x > 0.025),
        "BTC-ETH correlation > 0.6": ("btc_eth_corr_7d", lambda x: x > 0.6),
        "CUSUM bullish": ("cusum_bullish", lambda x: x == 1),
        "EMA ribbon bullish": ("ema_ribbon", lambda x: x == 1),
        "Markov P(high vol) > 0.5": ("markov_p_high_vol", lambda x: x > 0.5),
    }
    for name, (column, condition) in rules.items():
        valid = f[column].notna()
        on, off = valid & condition(f[column]), valid & ~condition(f[column])
        rows[(s, name)] = {"share of days ON": on.sum() / valid.sum(), "momentum payoff ON": payoff[on].mean(),
                           "momentum payoff OFF": payoff[off].mean(), "next-week return ON": t["fwd_ret"][on].mean(),
                           "next-week vol ON": t["fwd_vol"][on].mean(), "next-week vol OFF": t["fwd_vol"][off].mean()}
pd.DataFrame(rows).T

In [ ]:
def rsi_band_table(s, high=75, low=35):
    r, f = stretch[s]["rsi"], targets[s]["fwd_ret"]
    out = {}
    for label, mask in {f"RSI > {high}": r > high, f"RSI < {low}": r < low}.items():
        g = f[mask].groupby(f[mask].index.year)
        out[label] = pd.concat({"mean 7d return": g.mean(), "days": g.size()}, axis=1).unstack()
    return pd.DataFrame(out).T


for s in ASSETS:
    print(s, "- the report's RSI thresholds (75 / 35)")
    display(rsi_band_table(s).round(3))

In [ ]:
for s in ASSETS:
    v = reference[s]["vol_cluster"]
    print(s, "- PCA + K-means volatility clusters (0 = calmest)")
    display(pd.DataFrame({"share of days": v.value_counts(normalize=True),
                          "next-week vol": targets[s]["fwd_vol"].groupby(v).mean(),
                          "next-week return": targets[s]["fwd_ret"].groupby(v).mean()}).round(3))

In [ ]:
s = "BTCUSDT"
window = slice("2022-09-01", "2023-09-30")
d, f = daily[s].loc[window], reference[s].loc[window]
kal = ind.kalman_trend(daily[s]["close"]).loc[window]
gauss = ind.causal_gaussian(daily[s]["close"]).loc[window]
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
axes[0].plot(d["close"], color=COLORS[s], lw=1, label="BTC close")
axes[0].plot(kal["kalman_level"], color="black", lw=1, label="causal Kalman level")
axes[0].plot(gauss, color="tab:purple", lw=1, ls="--", label="one-sided Gaussian")
bull = f["cusum_bullish"] == 1
axes[0].fill_between(d.index, d["close"].min(), d["close"].max(), where=bull, color="green", alpha=0.08, label="CUSUM bullish")
axes[0].set_title("BTC: the report's reference-price filters and CUSUM regime (causal versions)")
axes[0].legend(loc="upper left", fontsize=8)
axes[1].plot(f["markov_p_high_vol"], color="tab:red", lw=1)
axes[1].set_ylabel("Markov P(high vol)")
plt.tight_layout()
plt.savefig(FIG / "ind_reference_filters.png", dpi=120)
plt.show()

**What it shows**
- **The ATR filter almost never filters BTC.**
  - On 70% of BTC days (and 53% of ETH days) most hours already have ATR below 1% of price; the median share of hours for BTC is 100%. Those days are calmer ahead (next-week vol 0.49 vs 0.79 for BTC).
  - Where it does apply, momentum paid *less*, not more. The share of hours with ATR < 1% has a negative return IC in 4 of 4 years for both coins.
  - An absolute threshold like "1% of price" means different things for BTC and ETH and in 2021 vs 2023. That is why our own volatility measures are relative to each coin's history.
- **The ATR > 2.5% exit rule** fires on very few days (27 BTC, 67 ETH in four years). On those days next-week volatility is extreme (≈ 1.2–1.5 annualised), but direction is mixed. It is a sensible risk flag, not a timing signal.
- **The BTC–ETH correlation filter barely binds.** The 7-day correlation is above 0.6 on about 98% of days (median 0.88), so "trade ETH only when correlated" is almost always true and changes little.
  - The correlation is informative about *risk* instead. When BTC and ETH decouple, next-week volatility ends up *higher* than current volatility suggests (beyond-vol IC negative in 8 of 8 coin-years).
- **Trend filters (EMA ribbon, Aroon, Heikin-Ashi, Kalman and Gaussian slopes, CUSUM) do not predict next-week direction.** Their return ICs change sign between years, as in section 5.
  - EMA-bullish and EMA-bearish BTC days were followed by the same average weekly return (0.6% vs 0.6%).
  - CUSUM-bullish BTC days averaged +0.9% next week vs +0.1% after bearish days, but the yearly ICs flip (+0.14, −0.11, +0.03, −0.08).
  - For ETH, the bullish readings (EMA ribbon, Kalman, CUSUM, Aroon) predict *higher* volatility beyond current vol in most years. This repeats section 8's finding that ETH rallies are violent.
- **The Markov-switching model** (filtered, refit yearly) finds a high-volatility regime on 41% of BTC days and 10% of ETH days. Next-week vol is clearly higher there (BTC 0.59 vs 0.47), but momentum payoff is about the same in both regimes for BTC. It measures volatility already visible in the data, much like our percentile rule.
- **PCA + K-means clusters are degenerate:** about 91% (BTC) and 94% (ETH) of days fall into one "normal" cluster, and the other clusters are tail days. Fat-tailed volatility does not form natural clusters, so the percentile-based regimes in sections 4 and 9 are more useful.
- **RSI 75 / 35:** RSI < 35 was followed by a positive average 7-day return for BTC in 4 of 4 years (+1.9%, +0.5%, +5.5%, +4.1%; 15–58 days a year). RSI > 75 was regime-dependent: −8.1% in 2021, +2.8% / +2.9% in 2023 / 2024, as with the 70 / 30 thresholds in section 6.

**Overall:** the report's indicators are all computable causally. They add two risk readings (extreme hourly ATR; BTC–ETH decoupling) and confirm the ETH upside-volatility asymmetry, but none of them gives a stable direction signal.

## 11. Conclusions: what to take into strategy design

| Indicator | Verdict | How to use it |
|---|---|---|
| Realised vol (hourly) | **Strong**: best forecaster of next-week vol | Core input for position sizing |
| ATR% / short-window range vol | **Strong** for daily-only data | Sizing fallback when only daily candles exist; prefer short windows |
| Smooth vs jump split | **Useful**: smooth vol persists, jumps fade | Size on the smooth part; don't over-react to one-off spikes |
| Down vs up semivariance | **Useful**: bad vol persists ~2× more | Shrink faster after sell-off volatility than after rally volatility |
| Vol percentile / squeeze | **Useful for size**: calm → expansion ahead | Reduce size or widen stops in a squeeze; never trade the squeeze for direction alone |
| Vol-of-vol, term structure | **Useful**: spikes mean-revert | Expect a storm to calm; avoid permanently de-risking after one spike |
| Relative volume | **Useful**: volume surge → more vol | Early-warning input for risk |
| Choppiness Index < 38.2 | **Moderate**: trend-following pays in clean trends | Conviction booster for trend entries |
| Volatility × efficiency regime | **Moderate**: momentum pays in calm trends | Full size in low-vol trend, reduced size in high-vol regimes |
| ADX | **Weak**: flags past moves, not future trend profit | Don't use as a trend switch |
| RSI / Stoch RSI / %B / z-score | **Weak**: direction depends on regime | At most a small dip-buying tilt inside an uptrend |
| Supertrend | **Weak**: no stable edge on daily bars | Not recommended |
| Drawdown from 90-day high | **Weak** dip-rebound tilt, **strong** vol warning | Shrink size in deep drawdowns; don't short them |
| CMF, taker imbalance | **None** for direction at a weekly horizon | Drop as direction signals |
| Hurst exponent | **None**: matches shuffled data (estimator bias) | Drop |
| Amihud illiquidity | **Promising but caveated**: return IC ≥ 0 in 8/8 coin-years | Re-test causally (volume level drifts, zero-fee period) before any use |
| *Reference report (section 10):* ATR % filter | **Weak**: an absolute 1% threshold barely filters BTC; > 2.5% flags extreme vol | Use relative (percentile) volatility instead |
| BTC–ETH correlation (7 d) | **Useful for risk**: decoupling → more vol ahead (8/8 coin-years); the 0.6 filter rarely binds | Risk warning, not a trade filter |
| EMA ribbon, Aroon, Heikin-Ashi, Kalman, causal Gaussian, CUSUM | **No direction edge**; ETH bullish readings → higher vol | Not as direction signals |
| Markov switching (filtered) | Tracks current volatility, like the percentile rule | Optional, descriptive |
| PCA + K-means clusters | **Degenerate** (≈ 91–94% in one cluster) | Drop |
| RSI < 35 | BTC positive next 7 d in 4/4 years (small samples) | At most a dip-buy tilt in an uptrend |

**Big picture.** Fancy indicators did not uncover a reliable hidden direction signal (the one candidate, illiquidity, needs a proper causal test). What they gave is a much richer **risk picture**: when storms are coming (squeeze, volume surge, bad semivariance), when they are passing (vol-of-vol, term-structure spikes), and when a trend is the orderly kind that keeps paying (low vol + high efficiency). This suggests strategies whose edge is **sizing and regime selection on top of a simple trend core**, rather than many direction signals.

*Caveats:* 7-day targets overlap, so neighbouring days are not independent; the yearly sign-consistency test is the guard against reading noise. Everything here is 2021–2024 only; 2025 was not used.